# Super-Point Computation


> **Inputs:**
> - Wave spectra in `.nc` format from CAWCR or ERA5.  
>   These can optionally include prior satellite correction.
>
> **Outputs:**
> - `superPoint.nc`: reconstructed offshore spectrum saved in the `outputs` folder.  
>   Used as input for BinWaves **Reconstruction** and later **Validation**.



To account for all possible wave energy and avoid the shadowing effect caused by land, a **Super-Point** is created. This approach, proposed by Cagigal et al. (2021), aggregates energy from multiple surrounding points near the study site.

<center>
    <img src="./assets/superPoint.png" width="80%">
</center>

In this case, the virtual **Super-Point** is constructed using two directional spectra from the CAWCR (Centre for Australian Weather and Climate Research) hindcast dataset (Durrant et al., 2014; Smith et al., 2020).


In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import os.path as op
import sys
import xarray as xr
import numpy as np

In [2]:
files = [
    'inputs/4469_spec_satellite_corrected.nc',
    'inputs/4466_spec_satellite_corrected.nc',
]
def load_station_data(ds: xr.Dataset) -> xr.Dataset:
    return ds.expand_dims("station")

# Use xr.open_mfdataset
stations_data = xr.open_mfdataset(
    files,
    combine='nested',
    concat_dim="station",
    preprocess=load_station_data,
)

# Define the actual station IDs
station_ids = [4469,4466]

# Replace the sequential station coordinates with actual station IDs
stations_data = stations_data.assign_coords(station=station_ids)
stations_data = stations_data.compute() 
stations_data

<xarray.Dataset> Size: 5GB
Dimensions:    (station: 2, time: 405963, freq: 29, dir: 24)
Coordinates:
  * time       (time) datetime64[ns] 3MB 1979-01-01 ... 2025-04-01
  * freq       (freq) float32 116B 0.035 0.0385 0.04235 ... 0.4171 0.4589 0.5047
  * dir        (dir) float32 96B 7.5 22.5 37.5 52.5 ... 307.5 322.5 337.5 352.5
  * station    (station) int64 16B 4469 4466
Data variables:
    efth       (station, time, freq, dir) float64 5GB 0.0 0.0 0.0 ... 0.0 0.0
    Depth      (station, time) float32 3MB 268.8 268.8 268.8 ... 41.14 41.14
    Wspeed     (station, time) float32 3MB 4.824 4.824 5.687 ... 10.44 11.17
    Wdir       (station, time) float32 3MB 334.2 334.2 340.0 ... 42.05 39.99
    longitude  (station, time) float32 3MB -74.84 -74.84 ... -77.74 -77.74
    latitude   (station, time) float32 3MB 36.61 36.61 36.61 ... 33.44 33.44

In [ ]:
import importlib
import bluemath_tk.waves.superpoint 
importlib.reload(bluemath_tk.waves.superpoint)
from bluemath_tk.waves.superpoint import superpoint_calculation

sectors_for_each_station = {}
stations_id = [4469,4466]
stations_sector = [(270,90), (90, 270)]

for i, station_id in enumerate(stations_data.station.values):
    sectors_for_each_station[station_id] = stations_sector[i]

superpoint_result = superpoint_calculation(
    stations_data=stations_data,
    stations_dimension_name='station',
    sectors_for_each_station=sectors_for_each_station,
    overlap_angle=22.5,
)
superpoint_result['latitude'] = 34.75
superpoint_result['longitude'] = -75.84
superpoint_result

In [ ]:
import importlib

from utils.plotting import Plot_spectrum
Plot_spectrum(superpoint_result, average=True);

In [ ]:
superpoint_result.to_netcdf("inputs/superPoint.nc")